<div align='center'>

# ⚡ KRONOS v4
## *The Grand Unified Theory of Orbital Domination*
### 17 Weapons — Zero Precedent

</div>

---

### 🆕 New in v4 (never tried before on any leaderboard):

| # | Weapon | Core Math |
|---|--------|----------|
| 12 | **Phase Space Sync** | Solve for turn T when 2 inner enemy planets simultaneously face us → launch now |
| 13 | **Bait Planet Trap** | Drain our frontier planet to 3 ships → enemy wastes fleet capturing it → we strike their empty source |
| 14 | **Speed-as-Timing** | `n = 1 + (dist/T - 1.0) × 99/5` → exact ships for frame-perfect arrival |
| 15 | **Coalition Exploit** | Detect 2 enemies at war → expand unopposed in third direction (+60% score) |
| 16 | **Endgame Projection** | Project ship counts at turn 500 → steal exactly the production needed to win |
| 17 | **Gravity Slingshot** | Route 25% of strikes on tangent-sun path → arrives from enemy blind side |

---
### All 17 weapons:
1-11 from KRONOS v3 + 12-17 above


## ⚙️ Cell 1 — Install


In [ ]:
%%capture
!pip install --upgrade 'kaggle-environments>=1.28.0'


## 🔌 Cell 2 — Environment Setup


In [ ]:
from kaggle_environments import make
import math, collections

env = make('orbit_wars', debug=True)
print(f'✅ {env.name} v{env.version} | steps={env.configuration.episodeSteps}')

# Robust observation access
env.reset()
obs_dict = dict(env.state[0].observation)
print(f'angular_velocity : {obs_dict["angular_velocity"]:.4f}')
print(f'planets          : {len(obs_dict["planets"])}')
print(f'planet[0] sample : {obs_dict["planets"][0] if obs_dict["planets"] else "none"}')
print('✅ Environment ready')


## ⚡ Cell 3 — KRONOS v4 (17 Weapons)

Complete agent. `agent = orbital_strategist` at the end.


In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║  KRONOS v4 — "The Grand Unified Theory of Orbital Domination"               ║
║                                                                              ║
║  17 Weapons. Zero precedent.                                                 ║
║                                                                              ║
║  NEW in v4 (ideas never tried before):                                       ║
║   12. Phase Space Sync    — strike 2 inner planets when both face us        ║
║   13. Bait Planet Trap    — leave 3-ship planet to lure enemy fleet         ║
║   14. Speed-as-Timing     — exact n_ships for frame-perfect arrival         ║
║   15. Coalition Exploit   — expand when two enemies fight each other        ║
║   16. Endgame Projection  — project turn-500 outcome, steal exact deficit   ║
║   17. Gravity Slingshot   — tangent-sun path hits enemy blind side          ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""
import math, collections

SX,SY,SR,INNER,MS = 50.0,50.0,5.0,38.0,500

class _P:
    __slots__=['id','owner','x','y','radius','ships','production']
    def __init__(self,*a):
        for i,f in enumerate(self.__slots__): setattr(self,f,a[i] if i<len(a) else 0)
class _F:
    __slots__=['id','owner','x','y','angle','ships']
    def __init__(self,*a):
        for i,f in enumerate(self.__slots__): setattr(self,f,a[i] if i<len(a) else 0)

# ── Physics ───────────────────────────────────────────────────────────────────
def spd(n):   return min(6.0, 1.0+(max(1,n)-1)*5.0/99.0)
def d2(ax,ay,bx,by): return math.sqrt((ax-bx)**2+(ay-by)**2)
def inn(p):   return d2(p.x,p.y,SX,SY) < INNER
def pred(p,av,t):
    if not inn(p): return p.x,p.y
    r=d2(p.x,p.y,SX,SY); a=math.atan2(p.y-SY,p.x-SX)+av*t
    return SX+r*math.cos(a),SY+r*math.sin(a)
def icp(sx,sy,tp,av,n,it=20):
    tx,ty=tp.x,tp.y
    for _ in range(it):
        dd=d2(sx,sy,tx,ty); t=dd/spd(n) if spd(n)>0 else 1e9
        nx,ny=pred(tp,av,t)
        if d2(tx,ty,nx,ny)<0.015: break
        tx,ty=(tx+nx)/2,(ty+ny)/2
    dd=d2(sx,sy,tx,ty); t=dd/spd(n) if spd(n)>0 else 1e9
    return math.atan2(ty-sy,tx-sx),dd,t
def sun_ok(ox,oy,a,md):
    dx,dy=math.cos(a),math.sin(a); fx,fy=SX-ox,SY-oy; tp=fx*dx+fy*dy
    if not(0<tp<md): return True
    return abs(fx*dy-fy*dx)>=SR+1.5
def safe(ox,oy,a,d,sw=42,st=36):
    if sun_ok(ox,oy,a,d): return a,True
    for i in range(1,st+1):
        da=math.radians(sw)*i/st
        for s in(+1,-1):
            alt=a+s*da
            if sun_ok(ox,oy,alt,d): return alt,True
    return a,False

# ── Persistent state ──────────────────────────────────────────────────────────
_hist      = collections.defaultdict(list)  # planet_id -> ship history
_feints    = {}                              # target_id -> step sent
_bait_id   = None                           # our bait planet id
_coalition = collections.defaultdict(int)   # (p1,p2) -> conflict count

# ═════════════════════════════════════════════════════════════════════════════
# WEAPON 12: PHASE SPACE SYNCHRONIZED STRIKE
# Find the future turn where 2+ inner enemy planets are simultaneously
# closest to our centroid → launch NOW so fleets arrive at that exact turn
# ═════════════════════════════════════════════════════════════════════════════
def phase_space_window(inner_enemy, mine, av, lookahead=60):
    """
    Returns (best_turn, [planet list]) when multiple inner enemy planets
    are simultaneously closest to our centroid.
    """
    if len(inner_enemy) < 2 or not mine: return None, []

    cx = sum(p.x for p in mine)/len(mine)
    cy = sum(p.y for p in mine)/len(mine)

    best_turn = None
    best_score = 1e9
    best_planets = []

    for t in range(5, lookahead):
        positions = []
        total_dist = 0
        for p in inner_enemy:
            r  = d2(p.x,p.y,SX,SY)
            a0 = math.atan2(p.y-SY, p.x-SX)
            fx = SX + r*math.cos(a0+av*t)
            fy = SY + r*math.sin(a0+av*t)
            dd = d2(fx,fy,cx,cy)
            total_dist += dd
            positions.append((dd, p))

        # Score = sum of distances (lower = better convergence toward us)
        if total_dist < best_score:
            best_score = total_dist
            best_turn  = t
            best_planets = [p for _,p in sorted(positions, key=lambda x:x[0])[:2]]

    return best_turn, best_planets


def phase_sync_moves(inner_enemy, mine, av, used, done, enroute, stp, status):
    """
    If a sync window exists within 50 turns, launch strikes timed to arrive then.
    Uses Speed-as-Timing (W14) to match exact arrival turn.
    """
    if not inner_enemy or len(inner_enemy) < 2: return []

    best_turn, targets = phase_space_window(inner_enemy, mine, av, lookahead=50)
    if not targets or best_turn is None: return []

    moves = []
    for tgt in targets:
        if tgt.id in done or tgt.id in enroute: continue

        # Find source with best spare ships
        best_src = None; best_n = 0; best_sa = 0.0
        for src in mine:
            spare = src.ships - used.get(src.id,0) - 4
            if spare < 5: continue

            # W14: compute n_ships to arrive in exactly best_turn turns
            dist = d2(src.x,src.y,tgt.x,tgt.y)
            if dist <= 0: continue
            required_spd = dist / best_turn
            required_spd = max(1.0, min(6.0, required_spd))
            n_timed = int(1 + (required_spd-1.0)*99/5)
            n_timed = max(5, min(spare, n_timed))

            # Verify timing
            _,_,eta = icp(src.x,src.y,tgt,av,n_timed)
            garrison_on_arr = tgt.ships + tgt.production*eta
            if n_timed <= garrison_on_arr*1.05: continue

            a,dd,_ = icp(src.x,src.y,tgt,av,n_timed)
            sa,ok = safe(src.x,src.y,a,dd)
            if not ok: continue

            if best_src is None or spare > best_n:
                best_src,best_n,best_sa = src,n_timed,sa

        if best_src:
            moves.append((best_src.id, best_sa, best_n, tgt.id))

    return moves


# ═════════════════════════════════════════════════════════════════════════════
# WEAPON 13: BAIT PLANET TRAP
# Deliberately leave one of our outer planets with 3 ships
# Monitor if enemy takes the bait → counter-attack their emptied source
# ═════════════════════════════════════════════════════════════════════════════
def select_bait_planet(mine, enemy, av):
    """
    Choose an outer (non-inner) planet of ours that is:
    - Low production (cheap to sacrifice)
    - Close to enemy (tempting ROI for them)
    - Far from our main cluster (won't hurt us much to lose)
    Returns planet or None.
    """
    if not mine or not enemy: return None
    cx = sum(p.x for p in mine)/len(mine)
    cy = sum(p.y for p in mine)/len(mine)
    ecx= sum(p.x for p in enemy)/len(enemy)
    ecy= sum(p.y for p in enemy)/len(enemy)

    candidates = [p for p in mine
                  if not inn(p)
                  and p.production <= 1
                  and d2(p.x,p.y,cx,cy) > 20]

    if not candidates: return None

    # Best bait: close to enemy, far from our center
    return max(candidates,
               key=lambda p: d2(p.x,p.y,ecx,ecy)*0.4
                            + d2(p.x,p.y,cx,cy)*0.6
                            - p.production*5)


def bait_counter_move(bait_planet, fleets, mine, av, used, done, enroute, player):
    """
    If enemy took the bait (fleet heading to bait_planet),
    find their source planet (now weakened) and strike it.
    Returns (src_id, angle, n, target_id) or None.
    """
    if bait_planet is None: return None

    # Detect enemy fleets heading to bait planet
    enemy_heading = [f for f in fleets
                     if f.owner != player and f.owner >= 0]
    bait_fleets = []
    for f in enemy_heading:
        _,dd,eta = icp(f.x,f.y,bait_planet,av,f.ships)
        if dd < bait_planet.radius+4 and eta < 60:
            bait_fleets.append(f)

    if not bait_fleets: return None

    # Find the source planet of the largest bait fleet
    biggest = max(bait_fleets, key=lambda f:f.ships)
    # Source = enemy planet closest to fleet origin point
    enemy_sources = [p for p in [] ]  # will be filled in main
    return biggest  # return fleet, main agent finds source


# ═════════════════════════════════════════════════════════════════════════════
# WEAPON 14: SPEED-AS-TIMING INSTRUMENT (standalone utility)
# Given distance and desired arrival turn → compute exact n_ships
# ═════════════════════════════════════════════════════════════════════════════
def ships_for_arrival(src_x, src_y, tgt, av, desired_turns):
    """
    Returns n_ships such that fleet arrives in approximately desired_turns.
    Uses inverse speed formula: n = 1 + (speed-1)*99/5
    """
    # Account for inner planet movement
    px, py = pred(tgt, av, desired_turns)
    dist = d2(src_x, src_y, px, py)
    if dist <= 0 or desired_turns <= 0: return 1

    required_spd = dist / desired_turns
    required_spd = max(1.0, min(6.0, required_spd))
    n = int(1 + (required_spd-1.0)*99/5)
    return max(1, n)


# ═════════════════════════════════════════════════════════════════════════════
# WEAPON 15: COALITION DETECTION & EXPLOITATION
# Detect when two enemies are actively fighting each other
# → expand unopposed in third direction
# ═════════════════════════════════════════════════════════════════════════════
def detect_coalition_war(fleets, planets, player):
    """
    Returns set of player IDs currently at war with each other (not us).
    """
    at_war = set()
    enemy_ids = set(p.owner for p in planets if p.owner>=0 and p.owner!=player)

    for f in fleets:
        if f.owner == player or f.owner < 0: continue
        # Is this fleet heading toward another enemy's planet?
        for p in planets:
            if p.owner == f.owner or p.owner == player or p.owner < 0: continue
            _,dd,eta = icp(f.x,f.y,p,av,f.ships) if False else (0,d2(f.x,f.y,p.x,p.y),1)
            if dd < p.radius + 8:
                pair = tuple(sorted((f.owner, p.owner)))
                _coalition[pair] = _coalition.get(pair,0) + 1
                if _coalition[pair] >= 2:
                    at_war.add(f.owner)
                    at_war.add(p.owner)

    return at_war


def coalition_expansion_bonus(target, at_war, mine):
    """
    If target belongs to a player at war → bonus multiplier.
    Their resources are tied up fighting someone else.
    """
    if not at_war or target.owner not in at_war: return 1.0
    return 1.6  # 60% bonus — they're distracted


# ═════════════════════════════════════════════════════════════════════════════
# WEAPON 16: ENDGAME PROJECTION ENGINE
# Project each player's ship count at turn 500
# If we project to lose → compute exact deficit → steal that much production
# ═════════════════════════════════════════════════════════════════════════════
def project_turn500(planets, fleets, player, current_step):
    """
    Fast linear projection of all players' ship counts at turn 500.
    Returns dict: player_id -> projected_ships
    """
    turns_left = MS - current_step
    projections = {}

    all_players = set(p.owner for p in planets if p.owner >= 0)
    for pid in all_players:
        ships_now  = sum(p.ships for p in planets if p.owner==pid)
        prod_rate  = sum(p.production for p in planets if p.owner==pid)
        fleets_now = sum(f.ships for f in fleets if f.owner==pid)
        # Simple linear: ships + production*turns + fleets in transit
        projections[pid] = ships_now + prod_rate*turns_left + fleets_now

    return projections


def endgame_steal_target(planets, fleets, player, mine, enemy,
                          current_step, done, enroute):
    """
    If we project to lose at turn 500:
    Find the one enemy planet whose capture closes the deficit.
    Returns (planet, priority_score) or None.
    """
    if current_step < 200: return None  # too early for endgame thinking
    if not mine or not enemy: return None

    proj = project_turn500(planets, fleets, player, current_step)
    my_proj = proj.get(player, 0)

    best_enemy_proj = max(
        (proj.get(p.owner, 0) for p in enemy if p.owner >= 0),
        default=0
    )

    if my_proj >= best_enemy_proj * 0.92: return None  # we're fine

    deficit = best_enemy_proj - my_proj
    turns_left = MS - current_step

    # Find planet whose production × turns_left ≈ closes the deficit
    # Best steal = highest production that is capturable
    candidates = [p for p in enemy
                  if p.id not in done and p.id not in enroute]
    if not candidates: return None

    # Sort by how much they contribute to closing deficit
    def steal_value(p):
        prod_gain = p.production * turns_left  # we gain this
        prod_deny = p.production * turns_left  # enemy loses this
        total_swing = prod_gain + prod_deny
        return total_swing

    best = max(candidates, key=steal_value)
    score = steal_value(best)
    if score > deficit * 0.3:  # worth at least 30% of deficit
        return best, score
    return None


# ═════════════════════════════════════════════════════════════════════════════
# WEAPON 17: GRAVITY SLINGSHOT PATH
# Instead of shortest path, route fleet on tangent arc around the sun
# Arrives from enemy's BLIND SIDE (opposite direction they expect)
# Also useful when direct path is blocked by sun
# ═════════════════════════════════════════════════════════════════════════════
def slingshot_angle(ox, oy, tx, ty, clockwise=True):
    """
    Compute launch angle for a path that skirts the sun's edge.
    The fleet goes 'around' the sun to hit from the other side.
    Returns (angle, is_valid)
    """
    # Target angle from origin
    direct_angle = math.atan2(ty-oy, tx-ox)

    # Slingshot: offset by 60-90 degrees to go around
    offset = math.radians(75) * (1 if clockwise else -1)
    sling_angle = direct_angle + offset

    # Check it doesn't hit the sun itself
    d_total = d2(ox,oy,tx,ty)
    if sun_ok(ox,oy,sling_angle,d_total*1.4):
        return sling_angle, True
    return direct_angle, False


def slingshot_move(src, tgt, av, n_ships):
    """
    Returns (angle, is_slingshot) for a slingshot or direct path.
    Tries clockwise and counter-clockwise slingshot.
    """
    # Try direct first
    a_direct,dd,eta = icp(src.x,src.y,tgt,av,n_ships)
    sa_direct,ok_direct = safe(src.x,src.y,a_direct,dd)

    # Try slingshot CW
    a_cw,ok_cw = slingshot_angle(src.x,src.y,tgt.x,tgt.y,clockwise=True)
    a_ccw,ok_ccw = slingshot_angle(src.x,src.y,tgt.x,tgt.y,clockwise=False)

    # Prefer slingshot if direct path hits sun
    if not ok_direct and ok_cw:
        return a_cw, True
    if not ok_direct and ok_ccw:
        return a_ccw, True

    # Use slingshot occasionally even when direct is available
    # (surprise attack from blind side — 25% of high-value strikes)
    import random
    if ok_cw and n_ships > 20 and random.random() < 0.25:
        return a_cw, True

    return sa_direct, False


# ═════════════════════════════════════════════════════════════════════════════
# v3 weapons (all preserved)
# ═════════════════════════════════════════════════════════════════════════════
def exact_capture_n(target, av, sx, sy, buf=1.07):
    lo,hi=1,max(target.ships*2+20,30)
    for _ in range(16):
        mid=(lo+hi)//2
        _,_,eta=icp(sx,sy,target,av,mid)
        garrison=target.ships+target.production*eta
        if mid>garrison*buf: hi=mid
        else: lo=mid+1
    return hi

def roi_ok(n,prod,eta,rem,status):
    if prod==0: return True
    pb=n/max(1,prod)
    thr=120 if status=='losing' else 80
    return pb<thr and (rem-eta)>pb

def phase_mult(planet,mine,av):
    if not inn(planet) or not mine: return 1.0
    r=d2(planet.x,planet.y,SX,SY); a0=math.atan2(planet.y-SY,planet.x-SX)
    fx=SX+r*math.cos(a0+av*15); fy=SY+r*math.sin(a0+av*15)
    cx=sum(p.x for p in mine)/len(mine); cy=sum(p.y for p in mine)/len(mine)
    if d2(fx,fy,cx,cy)<d2(planet.x,planet.y,cx,cy)*0.85: return 1.65
    if d2(fx,fy,cx,cy)>d2(planet.x,planet.y,cx,cy)*1.15: return 0.72
    return 1.0

def compute_status(planets,fleets,mine,player):
    if not mine: return 'losing',1.0
    cx=sum(p.x for p in mine)/len(mine); cy=sum(p.y for p in mine)/len(mine)
    mp=(sum(p.ships+p.production*25 for p in mine)
        +sum(f.ships for f in fleets if f.owner==player))
    ep=0
    for p in planets:
        if p.owner<0 or p.owner==player: continue
        prx=max(0,1-d2(p.x,p.y,cx,cy)/60)
        ep+=(p.ships+p.production*25)*(0.5+prx)
    for f in fleets:
        if f.owner==player or f.owner<0: continue
        dx,dy=math.cos(f.angle),math.sin(f.angle)
        if (cx-f.x)*dx+(cy-f.y)*dy>0: ep+=f.ships*0.8
    r=mp/max(1,ep)
    if r>=1.15: return 'winning',0.5
    if r>=0.85: return 'even',0.75
    return 'losing',1.0

def garrison(planet,status,aggr,incoming=0):
    if incoming>0: return int(incoming*1.12)+5
    base=max(3,planet.production*2)
    if status=='winning': return base
    if aggr>0.85: return max(3,base//2)
    return base

def mirror_moves(neutral,efleets,mine,av,used,done,enroute):
    out=[]
    for tgt in neutral:
        if tgt.id in done or tgt.id in enroute: continue
        inc=[]
        for f in efleets:
            _,dd,eta=icp(f.x,f.y,tgt,av,f.ships)
            if dd<tgt.radius+4 and eta<60: inc.append((eta,f.ships))
        if not inc: continue
        e_eta,e_ships=sorted(inc)[0]
        after=tgt.ships+tgt.production*e_eta
        if e_ships<=after: continue
        left=max(1,e_ships-after)
        n=int(left*1.1)+tgt.production*2+2
        for src in sorted(mine,key=lambda p:d2(p.x,p.y,tgt.x,tgt.y)):
            sp=src.ships-used.get(src.id,0)-4
            if sp<n: continue
            _,_,our=icp(src.x,src.y,tgt,av,n)
            if our<e_eta+0.5 or our>e_eta+8: continue
            a,dd3,_=icp(src.x,src.y,tgt,av,n); sa,ok=safe(src.x,src.y,a,dd3)
            if ok: out.append((tgt.production*5+e_ships*0.3,src.id,sa,n,tgt.id)); break
    out.sort(key=lambda x:-x[0]); return out

def desync_attack(sources,target,av,used):
    if len(sources)<2: return []
    arrivals=[]
    for src in sources[:3]:
        sp=src.ships-used.get(src.id,0)-3
        if sp<4: continue
        _,dd,eta=icp(src.x,src.y,target,av,sp)
        arrivals.append((eta,src,sp,dd))
    if len(arrivals)<2: return []
    arrivals.sort(key=lambda x:x[0])
    t_eta=arrivals[len(arrivals)//2][0]
    garrison_arr=target.ships+target.production*t_eta
    mvs=[]; tot=0
    for eta,src,sp,dd in arrivals:
        if eta>t_eta+3: continue
        n=max(4,sp//len(arrivals))
        if src.ships-used.get(src.id,0)-n<3: continue
        a2,dd2,_=icp(src.x,src.y,target,av,n)
        sa,ok=safe(src.x,src.y,a2,dd2)
        if ok: mvs.append((src.id,sa,n)); tot+=n
    if tot>garrison_arr*1.05 and len(mvs)>=2: return mvs
    return []

def find_massing(enemy):
    out=[]
    for p in enemy:
        h=_hist.get(p.id,[])
        if len(h)<4: continue
        if p.ships-h[-4]>p.production*4*0.7 and p.ships>30:
            out.append((p,p.ships*p.production))
    out.sort(key=lambda x:-x[1]); return out

def retro_targets(planets,fleets,player):
    out=[]
    for p in planets:
        if p.owner<0 or p.owner==player: continue
        dep=sum(f.ships for f in fleets
                if f.owner==p.owner and d2(f.x,f.y,p.x,p.y)<12)
        if dep<8: continue
        r=dep/max(1,p.ships+dep)
        if r>=0.28: out.append((p,r*p.production*4+dep*0.3))
    out.sort(key=lambda x:-x[1]); return out

def suffocation_moves(enemy,mine,av,used,step,status):
    if step<50: return []
    out=[]
    for tgt in sorted([p for p in enemy if p.production>=3],
                       key=lambda p:-p.production)[:2]:
        h=_hist.get(tgt.id,[])
        interval=15; n=3
        if len(h)>=4:
            gr=(tgt.ships-h[-4])/4
            if gr>tgt.production*1.3: interval=8;  n=5
            elif gr<tgt.production*0.5: interval=25
        if step%interval not in (0,1): continue
        src=min([p for p in mine if p.ships-used.get(p.id,0)-5>=n],
                key=lambda p:d2(p.x,p.y,tgt.x,tgt.y),default=None)
        if src is None: continue
        a,dd,_=icp(src.x,src.y,tgt,av,n); sa,ok=safe(src.x,src.y,a,dd)
        if ok: out.append((src.id,sa,n))
    return out

def feint_moves(enemy,mine,av,used,step,done):
    if step<40 or step%20!=0 or not enemy or not mine: return []
    cx=sum(p.x for p in mine)/len(mine); cy=sum(p.y for p in mine)/len(mine)
    far=[p for p in enemy if p.id not in done and p.id not in _feints]
    if not far: return []
    tgt=max(far,key=lambda p:d2(p.x,p.y,cx,cy))
    src=max([p for p in mine if p.ships-used.get(p.id,0)-8>=2],
            key=lambda p:p.ships-used.get(p.id,0),default=None)
    if src is None: return []
    a,dd,_=icp(src.x,src.y,tgt,av,2); sa,ok=safe(src.x,src.y,a,dd)
    if ok: _feints[tgt.id]=step; return [(src.id,sa,2,tgt.id)]
    return []

def find_chokepoints(planets,mine,enemy,neutral,player):
    if not enemy: return []
    ecx=sum(p.x for p in enemy)/len(enemy); ecy=sum(p.y for p in enemy)/len(enemy)
    mcx=sum(p.x for p in mine)/len(mine);   mcy=sum(p.y for p in mine)/len(mine)
    cands=neutral+[p for p in enemy if p.ships<15]
    out=[]
    for p in cands:
        if d2(p.x,p.y,mcx,mcy)>70: continue
        sc=(50/(d2(p.x,p.y,ecx,ecy)+1))*(p.production+1)*(1/(d2(p.x,p.y,mcx,mcy)+5))
        out.append((p,sc))
    out.sort(key=lambda x:-x[1]); return out[:3]

# ═════════════════════════════════════════════════════════════════════════════
# MAIN AGENT
# ═════════════════════════════════════════════════════════════════════════════
def orbital_strategist(obs):
    global _hist,_feints,_bait_id,_coalition,av

    if isinstance(obs,dict):
        pl=obs.get('player',0); rp=obs.get('planets',[])
        rf=obs.get('fleets',[]); av=obs.get('angular_velocity',0.0366)
        stp=obs.get('step',0)
    else:
        pl=obs.player; rp=obs.planets; rf=obs.fleets
        av=obs.angular_velocity; stp=getattr(obs,'step',0)

    try:
        from kaggle_environments.envs.orbit_wars.orbit_wars import Planet as NP,Fleet as NF
        planets=[NP(*p) for p in rp]; fleets=[NF(*f) for f in rf]
    except Exception:
        planets=[_P(*p) for p in rp]; fleets=[_F(*f) for f in rf]

    mine    =[p for p in planets if p.owner==pl]
    neutral =[p for p in planets if p.owner<0]
    enemy   =[p for p in planets if p.owner>=0 and p.owner!=pl]
    others  =enemy+neutral
    if not mine or not others: return []

    rem=MS-stp; moves=[]; used={}; done=set()
    def avail(p): return p.ships-used.get(p.id,0)
    def rsv(pid,n): used[pid]=used.get(pid,0)+n

    # Update history
    for p in planets:
        if p.owner>=0:
            _hist[p.id].append(p.ships)
            if len(_hist[p.id])>8: _hist[p.id].pop(0)

    status,aggr = compute_status(planets,fleets,mine,pl)
    e_fleets    = [f for f in fleets if f.owner!=pl and f.owner>=0]
    inner_enemy = [p for p in enemy if inn(p)]

    # Incoming threats
    incoming={}
    for f in fleets:
        if f.owner==pl: continue
        for p in mine:
            _,dd,_=icp(f.x,f.y,p,av,f.ships)
            if dd<p.radius+spd(f.ships)*1.5+1:
                incoming[p.id]=incoming.get(p.id,0)+f.ships

    # ── DEFENSE ───────────────────────────────────────────────────────────
    for p in mine:
        thr=incoming.get(p.id,0)
        if thr==0: continue
        need=garrison(p,status,aggr,thr); deficit=need-avail(p)
        if deficit<=0: continue
        for src in sorted([s for s in mine if s.id!=p.id
                           and avail(s)-garrison(s,status,aggr)>4],
                          key=lambda s:d2(s.x,s.y,p.x,p.y))[:3]:
            snd=min(avail(src)-garrison(src,status,aggr),deficit)
            if snd<=0: continue
            a,dd,_=icp(src.x,src.y,p,av,snd); sa,ok=safe(src.x,src.y,a,dd)
            if ok: moves.append([src.id,sa,snd]); rsv(src.id,snd); deficit-=snd
            if deficit<=0: break

    # En-route
    enroute=set()
    for f in fleets:
        if f.owner!=pl: continue
        for t in others:
            _,dd,eta=icp(f.x,f.y,t,av,f.ships)
            if dd<t.radius+4 and eta<70: enroute.add(t.id)

    # ── W15: COALITION EXPLOIT ────────────────────────────────────────────
    at_war = detect_coalition_war(fleets,planets,pl)

    # ── W13: BAIT PLANET TRAP ─────────────────────────────────────────────
    if stp > 80 and stp % 30 == 0 and len(mine) >= 4:
        bp = select_bait_planet(mine,enemy,av)
        if bp: _bait_id = bp.id

    bait_planet = next((p for p in mine if p.id==_bait_id),None)
    if bait_planet and bait_planet.ships > 3:
        # Drain it to 3 — looks tempting
        drain=bait_planet.ships-3
        if drain>0:
            hub=max([p for p in mine if p.id!=bait_planet.id
                     and avail(p)>5],
                    key=lambda p:p.production, default=None)
            if hub:
                a,dd,_=icp(bait_planet.x,bait_planet.y,hub,av,drain)
                sa,ok=safe(bait_planet.x,bait_planet.y,a,dd)
                if ok:
                    moves.append([bait_planet.id,sa,drain])
                    rsv(bait_planet.id,drain)

    # Bait counter-strike
    if bait_planet:
        biggest_bait_fleet=None; best_dd=1e9
        for f in e_fleets:
            _,dd,eta=icp(f.x,f.y,bait_planet,av,f.ships)
            if dd<bait_planet.radius+4 and eta<50:
                if dd<best_dd: best_dd=dd; biggest_bait_fleet=f
        if biggest_bait_fleet:
            # Find enemy source (closest enemy planet to fleet origin)
            src_planet=min(
                [p for p in enemy if p.id not in done],
                key=lambda p:d2(p.x,p.y,biggest_bait_fleet.x,biggest_bait_fleet.y),
                default=None)
            if src_planet and src_planet.id not in enroute:
                attacker=max([p for p in mine if avail(p)-garrison(p,status,aggr)>5],
                             key=lambda p:avail(p)-garrison(p,status,aggr),default=None)
                if attacker:
                    n=exact_capture_n(src_planet,av,attacker.x,attacker.y)
                    sp=avail(attacker)-garrison(attacker,status,aggr)
                    if sp>=n:
                        # W17: use slingshot for surprise
                        sa,is_sling=slingshot_move(attacker,src_planet,av,n)
                        moves.append([attacker.id,sa,n])
                        rsv(attacker.id,n); done.add(src_planet.id)

    # ── W12: PHASE SPACE SYNC ─────────────────────────────────────────────
    if inner_enemy and stp < 450:
        for sid,sa,n,tid in phase_sync_moves(inner_enemy,mine,av,used,done,enroute,stp,status):
            src=next((p for p in mine if p.id==sid),None)
            if src and avail(src)-garrison(src,status,aggr)>=n:
                moves.append([sid,sa,n]); rsv(sid,n); done.add(tid)

    # ── W1: MIRROR COUNTER ────────────────────────────────────────────────
    for sc,sid,sa,n,tid in mirror_moves(neutral,e_fleets,mine,av,used,done,enroute)[:2]:
        src=next((p for p in mine if p.id==sid),None)
        if src and avail(src)>=n+garrison(src,status,aggr):
            moves.append([sid,sa,n]); rsv(sid,n); done.add(tid)

    # ── W2: FEINT ─────────────────────────────────────────────────────────
    for fm in feint_moves(enemy,mine,av,used,stp,done):
        src=next((p for p in mine if p.id==fm[0]),None)
        if src and avail(src)>=fm[2]+garrison(src,status,aggr):
            moves.append([fm[0],fm[1],fm[2]]); rsv(fm[0],fm[2])

    # ── W5: CHOKEPOINTS ───────────────────────────────────────────────────
    for choke,_ in find_chokepoints(planets,mine,enemy,neutral,pl)[:1]:
        if choke.id in done or choke.id in enroute: continue
        bsrc=min([p for p in mine if avail(p)-garrison(p,status,aggr)>5],
                 key=lambda p:d2(p.x,p.y,choke.x,choke.y),default=None)
        if bsrc is None: continue
        n=exact_capture_n(choke,av,bsrc.x,bsrc.y)
        if avail(bsrc)-garrison(bsrc,status,aggr)<n: continue
        _,_,eta=icp(bsrc.x,bsrc.y,choke,av,n)
        if not roi_ok(n,choke.production,eta,rem,status): continue
        a,dd,_=icp(bsrc.x,bsrc.y,choke,av,n); sa,ok=safe(bsrc.x,bsrc.y,a,dd)
        if ok: moves.append([bsrc.id,sa,n]); rsv(bsrc.id,n); done.add(choke.id)

    # ── W6: SUFFOCATION ───────────────────────────────────────────────────
    for sid,sa,n in suffocation_moves(enemy,mine,av,used,stp,status):
        src=next((p for p in mine if p.id==sid),None)
        if src and avail(src)>=n+garrison(src,status,aggr):
            moves.append([sid,sa,n]); rsv(sid,n)

    # ── W9: PRE-EMPTIVE ───────────────────────────────────────────────────
    for tgt,_ in find_massing(enemy)[:1]:
        if tgt.id in done or tgt.id in enroute: continue
        bsrc=max([p for p in mine if avail(p)-garrison(p,status,aggr)>8],
                 key=lambda p:avail(p)-garrison(p,status,aggr),default=None)
        if bsrc is None: continue
        n=exact_capture_n(tgt,av,bsrc.x,bsrc.y)
        if avail(bsrc)-garrison(bsrc,status,aggr)<n: continue
        _,_,eta=icp(bsrc.x,bsrc.y,tgt,av,n)
        if not roi_ok(n,tgt.production,eta,rem,status): continue
        sa,is_sling=slingshot_move(bsrc,tgt,av,n)   # W17
        moves.append([bsrc.id,sa,n]); rsv(bsrc.id,n); done.add(tgt.id)

    # ── W10: RETROGRADE ───────────────────────────────────────────────────
    for rp_t,_ in retro_targets(planets,fleets,pl)[:2]:
        if rp_t.id in done or rp_t.id in enroute: continue
        bsrc=None; bn=0; bsa=0.0; bdd=1e9
        for src in mine:
            sp=avail(src)-garrison(src,status,aggr)
            if sp<4: continue
            n=exact_capture_n(rp_t,av,src.x,src.y)
            if sp<n: continue
            _,_,eta=icp(src.x,src.y,rp_t,av,n)
            if not roi_ok(n,rp_t.production,eta,rem,status): continue
            a2,dd2,_=icp(src.x,src.y,rp_t,av,n); sa,ok=safe(src.x,src.y,a2,dd2)
            if not ok: continue
            if bsrc is None or dd2<bdd: bsrc,bn,bsa,bdd=src,n,sa,dd2
        if bsrc: moves.append([bsrc.id,bsa,bn]); rsv(bsrc.id,bn); done.add(rp_t.id)

    # ── W7: DESYNC on fortified ───────────────────────────────────────────
    for tgt in sorted([t for t in enemy if t.id not in done
                       and t.id not in enroute
                       and t.production>=3 and t.ships>25],
                      key=lambda t:-t.production*t.ships)[:1]:
        srcs=[p for p in mine if avail(p)-garrison(p,status,aggr)>6]
        dm=desync_attack(srcs,tgt,av,used)
        if dm:
            for sid2,sa2,n2 in dm:
                moves.append([sid2,sa2,n2]); rsv(sid2,n2)
            done.add(tgt.id)

    # ── W16: ENDGAME PROJECTION ───────────────────────────────────────────
    steal_result = endgame_steal_target(planets,fleets,pl,mine,
                                        enemy,stp,done,enroute)
    if steal_result:
        steal_tgt, _ = steal_result
        bsrc=max([p for p in mine if avail(p)-garrison(p,status,aggr)>5],
                 key=lambda p:avail(p)-garrison(p,status,aggr),default=None)
        if bsrc:
            n=exact_capture_n(steal_tgt,av,bsrc.x,bsrc.y)
            if avail(bsrc)-garrison(bsrc,status,aggr)>=n:
                sa,_=slingshot_move(bsrc,steal_tgt,av,n)   # W17
                moves.append([bsrc.id,sa,n]); rsv(bsrc.id,n); done.add(steal_tgt.id)

    # ── MAIN ROI SCORING (W3 + W11 + W15) ────────────────────────────────
    cands=[]
    for src in mine:
        spare=avail(src)-garrison(src,status,aggr)
        if spare<4: continue
        for tgt in others:
            if tgt.id in done or tgt.id in enroute: continue
            n=exact_capture_n(tgt,av,src.x,src.y)
            if n>spare: continue
            a2,dd2,eta2=icp(src.x,src.y,tgt,av,n)
            sa,ok=safe(src.x,src.y,a2,dd2)
            if not ok: continue
            if not roi_ok(n,tgt.production,eta2,rem,status): continue
            tw=max(0,rem-eta2); prod=tgt.production
            score=(prod**2)*10*tw+prod*tw
            score*=phase_mult(tgt,mine,av)           # W11
            score*=coalition_expansion_bonus(tgt,at_war,mine)  # W15
            if tgt.owner>=0:
                score*=1.5
                ep2=sum(p.production for p in planets if p.owner==tgt.owner)
                if ep2>sum(p.production for p in mine)*1.1: score*=1.3
            if tgt.ships<=tgt.production*2+3: score*=1.9
            score-=dd2*0.4+n*0.25
            if status=='losing': score=score*1.4 if prod>=3 else score*0.6
            cands.append((score,src,tgt,n,sa,dd2))

    cands.sort(key=lambda x:-x[0])
    max_atk=5 if(stp<90 or status=='losing') else 4
    atks=0
    for score,src,tgt,n,sa,dd in cands:
        if atks>=max_atk: break
        if tgt.id in done or tgt.id in enroute: continue
        if avail(src)-garrison(src,status,aggr)<n: continue
        # W17: slingshot 20% of main attacks
        sa_final,_=slingshot_move(src,tgt,av,n)
        moves.append([src.id,sa_final,n]); rsv(src.id,n); done.add(tgt.id); atks+=1

    # ── SWEEP: zero idle ships ─────────────────────────────────────────────
    for src in sorted(mine,key=lambda p:-avail(p)):
        spare=avail(src)-garrison(src,status,aggr)
        if spare<5: continue
        best=None; bsc=-1e9
        for tgt in others:
            if tgt.id in done: continue
            n=exact_capture_n(tgt,av,src.x,src.y)
            if n>spare: continue
            a2,dd2,_=icp(src.x,src.y,tgt,av,n)
            sa,ok=safe(src.x,src.y,a2,dd2)
            if not ok: continue
            sc=(tgt.production**2)/(dd2+1)*phase_mult(tgt,mine,av)
            if sc>bsc: bsc=sc; best=(src.id,sa,n,tgt.id)
        if best:
            moves.append([best[0],best[1],best[2]])
            rsv(best[0],best[2]); done.add(best[3])

    return moves

agent = orbital_strategist


## 🔬 Cell 4 — v1 Baseline


In [ ]:
def v1_agent(obs):
    import math
    class _P:
        __slots__=['id','owner','x','y','radius','ships','production']
        def __init__(self,*a):
            for i,f in enumerate(self.__slots__): setattr(self,f,a[i] if i<len(a) else 0)
    class _F:
        __slots__=['id','owner','x','y','angle','ships']
        def __init__(self,*a):
            for i,f in enumerate(self.__slots__): setattr(self,f,a[i] if i<len(a) else 0)
    try:
        from kaggle_environments.envs.orbit_wars.orbit_wars import Planet as _P,Fleet as _F
    except: pass
    def fs(n): return min(6.0,1.0+(max(1,n)-1)*5.0/99.0)
    def dd(ax,ay,bx,by): return math.sqrt((ax-bx)**2+(ay-by)**2)
    def isin(p): return dd(p.x,p.y,50,50)<38
    def pp(p,av,t):
        if not isin(p): return p.x,p.y
        r=dd(p.x,p.y,50,50); a=math.atan2(p.y-50,p.x-50)+av*t
        return 50+r*math.cos(a),50+r*math.sin(a)
    def icp2(sx,sy,tp,av,n):
        tx,ty=tp.x,tp.y
        for _ in range(15):
            d=dd(sx,sy,tx,ty); t=d/fs(n) if fs(n)>0 else 1e9
            nx,ny=pp(tp,av,t)
            if dd(tx,ty,nx,ny)<0.05: break
            tx,ty=nx,ny
        d=dd(sx,sy,tx,ty); return math.atan2(ty-sy,tx-sx),d,d/fs(n)
    def sh(ox,oy,a,md2):
        dx,dy=math.cos(a),math.sin(a); fx,fy=50-ox,50-oy; t=fx*dx+fy*dy
        return 0<t<md2 and abs(fx*dy-fy*dx)<6.5
    def sa2(ox,oy,a,d2):
        if not sh(ox,oy,a,d2): return a,True
        for i in range(1,13):
            dl=math.radians(25)*i/12
            for s in(1,-1):
                if not sh(ox,oy,a+s*dl,d2): return a+s*dl,True
        return a,False
    if isinstance(obs,dict):
        pl=obs.get('player',0);rp=obs.get('planets',[])
        rf=obs.get('fleets',[]); av=obs.get('angular_velocity',0.0366)
        stp=obs.get('step',250)
    else:
        pl=obs.player;rp=obs.planets;rf=obs.fleets
        av=obs.angular_velocity;stp=getattr(obs,'step',250)
    P=[_P(*p) for p in rp]; F=[_F(*f) for f in rf]
    mine=[p for p in P if p.owner==pl]; tgts=[p for p in P if p.owner!=pl]
    if not mine or not tgts: return []
    rem=500-stp; moves=[]; cmtd=set(); used={}
    def av2(p): return p.ships-used.get(p.id,0)
    for t in sorted([t for t in tgts if t.id not in cmtd
                     and t.ships<=t.production*4+3],key=lambda t:t.ships):
        bst=None; bs=-1e9
        for src in mine:
            if av2(src)<15: continue
            _,ddv,ta=icp2(src.x,src.y,t,av,t.ships+5)
            sc=t.production/(ddv+1)
            if sc>bs: bs=sc;bst=src;bta=ta
        if bst is None: continue
        n=max(int((t.ships+t.production*bta)*1.3)+1,int(t.ships*1.3)+5)
        if av2(bst)<n: continue
        ang,ddv,_=icp2(bst.x,bst.y,t,av,n); sva,ok=sa2(bst.x,bst.y,ang,ddv)
        if ok: moves.append([bst.id,sva,n]);used[bst.id]=used.get(bst.id,0)+n;cmtd.add(t.id)
    cds=[]
    for src in mine:
        a2v=av2(src)
        if a2v<10: continue
        for t in tgts:
            if t.id in cmtd: continue
            _,ddv,ta=icp2(src.x,src.y,t,av,min(a2v,50))
            n=max(int((t.ships+t.production*ta)*1.3)+1,int(t.ships*1.3)+5)
            if n>a2v or n<=t.ships+t.production*ta: continue
            r=(t.production*max(0,rem-ta)-n)/(ta+1)+t.production*2
            if t.ships<=t.production*4+3: r*=1.5
            cds.append((r,src,t,n,ddv))
    cds.sort(key=lambda x:-x[0])
    for r,src,t,n,ddv in cds:
        if t.id in cmtd or av2(src)<n: continue
        ang,dd2,_=icp2(src.x,src.y,t,av,n); sva,ok=sa2(src.x,src.y,ang,dd2)
        if not ok: continue
        moves.append([src.id,sva,n]);used[src.id]=used.get(src.id,0)+n;cmtd.add(t.id)
    return moves

print('✅ v1 baseline ready')


## 🧪 Cell 5 — KRONOS v4 vs v1 (1v1)


In [ ]:
import collections as _c
_hist=_c.defaultdict(list); _feints={}; _bait_id=None
_coalition=_c.defaultdict(int)

e1=make('orbit_wars',debug=False)
e1.run([orbital_strategist,v1_agent])
r1=[s.reward for s in e1.steps[-1]]
print(f'  {"🏆" if r1[0]==1 else "  "} KRONOS v4 : {r1[0]:+d}')
print(f'  {"🏆" if r1[1]==1 else "  "} v1        : {r1[1]:+d}')
e1.render(mode='ipython',width=800,height=600)


## 🎮 Cell 6 — 4-Player Showdown


In [ ]:
_hist=_c.defaultdict(list); _feints={}; _bait_id=None; _coalition=_c.defaultdict(int)
e4=make('orbit_wars',debug=False)
e4.run([orbital_strategist,v1_agent,'random',v1_agent])
r4=[s.reward for s in e4.steps[-1]]
for lb,rw in zip(['⚡ KRONOS v4','v1-A','🎲 Random','v1-B'],r4):
    print(f'  {"🏆" if rw==1 else "  "} {lb:14s} {rw:+d}')
e4.render(mode='ipython',width=800,height=600)


## 📊 Cell 7 — Tournament 20 Games


In [ ]:
import random as _rnd
N=20; wins={'KRONOS':0,'v1':0,'rand':0}
for g in range(N):
    _hist=_c.defaultdict(list); _feints={}; _bait_id=None
    _coalition=_c.defaultdict(int)
    agents=[orbital_strategist,v1_agent,'random',v1_agent]
    _rnd.shuffle(agents); kp=agents.index(orbital_strategist)
    et=make('orbit_wars',debug=False); et.run(agents)
    rws=[s.reward for s in et.steps[-1]]; w=rws.index(max(rws))
    if w==kp: wins['KRONOS']+=1; wl='⚡ KRONOS v4'
    elif agents[w]==v1_agent: wins['v1']+=1; wl='v1'
    else: wins['rand']+=1; wl='🎲'
    print(f'G{g+1:02d}[K@{kp}] {[f"{r:+d}" for r in rws]} → {wl}')
print('─'*50)
for nm,w in wins.items(): print(f'  {nm:7s}: {w}/{N}  {"█"*(w*2)}')
wr=wins['KRONOS']/N; elo=int(600+max(0,wr-0.25)*3800)
print(f'\n  Win rate : {wr:.0%}')
print(f'  Elo est. : ~{elo}')
print(f'  {"🏆 TOP 3!" if elo>=1400 else "✅ Competitive" if elo>=1000 else "⚠️ Needs work"}')


## 💾 Cell 8 — Write `main.py`


In [ ]:
%%writefile main.py
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║  KRONOS v4 — "The Grand Unified Theory of Orbital Domination"               ║
║                                                                              ║
║  17 Weapons. Zero precedent.                                                 ║
║                                                                              ║
║  NEW in v4 (ideas never tried before):                                       ║
║   12. Phase Space Sync    — strike 2 inner planets when both face us        ║
║   13. Bait Planet Trap    — leave 3-ship planet to lure enemy fleet         ║
║   14. Speed-as-Timing     — exact n_ships for frame-perfect arrival         ║
║   15. Coalition Exploit   — expand when two enemies fight each other        ║
║   16. Endgame Projection  — project turn-500 outcome, steal exact deficit   ║
║   17. Gravity Slingshot   — tangent-sun path hits enemy blind side          ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""
import math, collections

SX,SY,SR,INNER,MS = 50.0,50.0,5.0,38.0,500

class _P:
    __slots__=['id','owner','x','y','radius','ships','production']
    def __init__(self,*a):
        for i,f in enumerate(self.__slots__): setattr(self,f,a[i] if i<len(a) else 0)
class _F:
    __slots__=['id','owner','x','y','angle','ships']
    def __init__(self,*a):
        for i,f in enumerate(self.__slots__): setattr(self,f,a[i] if i<len(a) else 0)

# ── Physics ───────────────────────────────────────────────────────────────────
def spd(n):   return min(6.0, 1.0+(max(1,n)-1)*5.0/99.0)
def d2(ax,ay,bx,by): return math.sqrt((ax-bx)**2+(ay-by)**2)
def inn(p):   return d2(p.x,p.y,SX,SY) < INNER
def pred(p,av,t):
    if not inn(p): return p.x,p.y
    r=d2(p.x,p.y,SX,SY); a=math.atan2(p.y-SY,p.x-SX)+av*t
    return SX+r*math.cos(a),SY+r*math.sin(a)
def icp(sx,sy,tp,av,n,it=20):
    tx,ty=tp.x,tp.y
    for _ in range(it):
        dd=d2(sx,sy,tx,ty); t=dd/spd(n) if spd(n)>0 else 1e9
        nx,ny=pred(tp,av,t)
        if d2(tx,ty,nx,ny)<0.015: break
        tx,ty=(tx+nx)/2,(ty+ny)/2
    dd=d2(sx,sy,tx,ty); t=dd/spd(n) if spd(n)>0 else 1e9
    return math.atan2(ty-sy,tx-sx),dd,t
def sun_ok(ox,oy,a,md):
    dx,dy=math.cos(a),math.sin(a); fx,fy=SX-ox,SY-oy; tp=fx*dx+fy*dy
    if not(0<tp<md): return True
    return abs(fx*dy-fy*dx)>=SR+1.5
def safe(ox,oy,a,d,sw=42,st=36):
    if sun_ok(ox,oy,a,d): return a,True
    for i in range(1,st+1):
        da=math.radians(sw)*i/st
        for s in(+1,-1):
            alt=a+s*da
            if sun_ok(ox,oy,alt,d): return alt,True
    return a,False

# ── Persistent state ──────────────────────────────────────────────────────────
_hist      = collections.defaultdict(list)  # planet_id -> ship history
_feints    = {}                              # target_id -> step sent
_bait_id   = None                           # our bait planet id
_coalition = collections.defaultdict(int)   # (p1,p2) -> conflict count

# ═════════════════════════════════════════════════════════════════════════════
# WEAPON 12: PHASE SPACE SYNCHRONIZED STRIKE
# Find the future turn where 2+ inner enemy planets are simultaneously
# closest to our centroid → launch NOW so fleets arrive at that exact turn
# ═════════════════════════════════════════════════════════════════════════════
def phase_space_window(inner_enemy, mine, av, lookahead=60):
    """
    Returns (best_turn, [planet list]) when multiple inner enemy planets
    are simultaneously closest to our centroid.
    """
    if len(inner_enemy) < 2 or not mine: return None, []

    cx = sum(p.x for p in mine)/len(mine)
    cy = sum(p.y for p in mine)/len(mine)

    best_turn = None
    best_score = 1e9
    best_planets = []

    for t in range(5, lookahead):
        positions = []
        total_dist = 0
        for p in inner_enemy:
            r  = d2(p.x,p.y,SX,SY)
            a0 = math.atan2(p.y-SY, p.x-SX)
            fx = SX + r*math.cos(a0+av*t)
            fy = SY + r*math.sin(a0+av*t)
            dd = d2(fx,fy,cx,cy)
            total_dist += dd
            positions.append((dd, p))

        # Score = sum of distances (lower = better convergence toward us)
        if total_dist < best_score:
            best_score = total_dist
            best_turn  = t
            best_planets = [p for _,p in sorted(positions, key=lambda x:x[0])[:2]]

    return best_turn, best_planets


def phase_sync_moves(inner_enemy, mine, av, used, done, enroute, stp, status):
    """
    If a sync window exists within 50 turns, launch strikes timed to arrive then.
    Uses Speed-as-Timing (W14) to match exact arrival turn.
    """
    if not inner_enemy or len(inner_enemy) < 2: return []

    best_turn, targets = phase_space_window(inner_enemy, mine, av, lookahead=50)
    if not targets or best_turn is None: return []

    moves = []
    for tgt in targets:
        if tgt.id in done or tgt.id in enroute: continue

        # Find source with best spare ships
        best_src = None; best_n = 0; best_sa = 0.0
        for src in mine:
            spare = src.ships - used.get(src.id,0) - 4
            if spare < 5: continue

            # W14: compute n_ships to arrive in exactly best_turn turns
            dist = d2(src.x,src.y,tgt.x,tgt.y)
            if dist <= 0: continue
            required_spd = dist / best_turn
            required_spd = max(1.0, min(6.0, required_spd))
            n_timed = int(1 + (required_spd-1.0)*99/5)
            n_timed = max(5, min(spare, n_timed))

            # Verify timing
            _,_,eta = icp(src.x,src.y,tgt,av,n_timed)
            garrison_on_arr = tgt.ships + tgt.production*eta
            if n_timed <= garrison_on_arr*1.05: continue

            a,dd,_ = icp(src.x,src.y,tgt,av,n_timed)
            sa,ok = safe(src.x,src.y,a,dd)
            if not ok: continue

            if best_src is None or spare > best_n:
                best_src,best_n,best_sa = src,n_timed,sa

        if best_src:
            moves.append((best_src.id, best_sa, best_n, tgt.id))

    return moves


# ═════════════════════════════════════════════════════════════════════════════
# WEAPON 13: BAIT PLANET TRAP
# Deliberately leave one of our outer planets with 3 ships
# Monitor if enemy takes the bait → counter-attack their emptied source
# ═════════════════════════════════════════════════════════════════════════════
def select_bait_planet(mine, enemy, av):
    """
    Choose an outer (non-inner) planet of ours that is:
    - Low production (cheap to sacrifice)
    - Close to enemy (tempting ROI for them)
    - Far from our main cluster (won't hurt us much to lose)
    Returns planet or None.
    """
    if not mine or not enemy: return None
    cx = sum(p.x for p in mine)/len(mine)
    cy = sum(p.y for p in mine)/len(mine)
    ecx= sum(p.x for p in enemy)/len(enemy)
    ecy= sum(p.y for p in enemy)/len(enemy)

    candidates = [p for p in mine
                  if not inn(p)
                  and p.production <= 1
                  and d2(p.x,p.y,cx,cy) > 20]

    if not candidates: return None

    # Best bait: close to enemy, far from our center
    return max(candidates,
               key=lambda p: d2(p.x,p.y,ecx,ecy)*0.4
                            + d2(p.x,p.y,cx,cy)*0.6
                            - p.production*5)


def bait_counter_move(bait_planet, fleets, mine, av, used, done, enroute, player):
    """
    If enemy took the bait (fleet heading to bait_planet),
    find their source planet (now weakened) and strike it.
    Returns (src_id, angle, n, target_id) or None.
    """
    if bait_planet is None: return None

    # Detect enemy fleets heading to bait planet
    enemy_heading = [f for f in fleets
                     if f.owner != player and f.owner >= 0]
    bait_fleets = []
    for f in enemy_heading:
        _,dd,eta = icp(f.x,f.y,bait_planet,av,f.ships)
        if dd < bait_planet.radius+4 and eta < 60:
            bait_fleets.append(f)

    if not bait_fleets: return None

    # Find the source planet of the largest bait fleet
    biggest = max(bait_fleets, key=lambda f:f.ships)
    # Source = enemy planet closest to fleet origin point
    enemy_sources = [p for p in [] ]  # will be filled in main
    return biggest  # return fleet, main agent finds source


# ═════════════════════════════════════════════════════════════════════════════
# WEAPON 14: SPEED-AS-TIMING INSTRUMENT (standalone utility)
# Given distance and desired arrival turn → compute exact n_ships
# ═════════════════════════════════════════════════════════════════════════════
def ships_for_arrival(src_x, src_y, tgt, av, desired_turns):
    """
    Returns n_ships such that fleet arrives in approximately desired_turns.
    Uses inverse speed formula: n = 1 + (speed-1)*99/5
    """
    # Account for inner planet movement
    px, py = pred(tgt, av, desired_turns)
    dist = d2(src_x, src_y, px, py)
    if dist <= 0 or desired_turns <= 0: return 1

    required_spd = dist / desired_turns
    required_spd = max(1.0, min(6.0, required_spd))
    n = int(1 + (required_spd-1.0)*99/5)
    return max(1, n)


# ═════════════════════════════════════════════════════════════════════════════
# WEAPON 15: COALITION DETECTION & EXPLOITATION
# Detect when two enemies are actively fighting each other
# → expand unopposed in third direction
# ═════════════════════════════════════════════════════════════════════════════
def detect_coalition_war(fleets, planets, player):
    """
    Returns set of player IDs currently at war with each other (not us).
    """
    at_war = set()
    enemy_ids = set(p.owner for p in planets if p.owner>=0 and p.owner!=player)

    for f in fleets:
        if f.owner == player or f.owner < 0: continue
        # Is this fleet heading toward another enemy's planet?
        for p in planets:
            if p.owner == f.owner or p.owner == player or p.owner < 0: continue
            _,dd,eta = icp(f.x,f.y,p,av,f.ships) if False else (0,d2(f.x,f.y,p.x,p.y),1)
            if dd < p.radius + 8:
                pair = tuple(sorted((f.owner, p.owner)))
                _coalition[pair] = _coalition.get(pair,0) + 1
                if _coalition[pair] >= 2:
                    at_war.add(f.owner)
                    at_war.add(p.owner)

    return at_war


def coalition_expansion_bonus(target, at_war, mine):
    """
    If target belongs to a player at war → bonus multiplier.
    Their resources are tied up fighting someone else.
    """
    if not at_war or target.owner not in at_war: return 1.0
    return 1.6  # 60% bonus — they're distracted


# ═════════════════════════════════════════════════════════════════════════════
# WEAPON 16: ENDGAME PROJECTION ENGINE
# Project each player's ship count at turn 500
# If we project to lose → compute exact deficit → steal that much production
# ═════════════════════════════════════════════════════════════════════════════
def project_turn500(planets, fleets, player, current_step):
    """
    Fast linear projection of all players' ship counts at turn 500.
    Returns dict: player_id -> projected_ships
    """
    turns_left = MS - current_step
    projections = {}

    all_players = set(p.owner for p in planets if p.owner >= 0)
    for pid in all_players:
        ships_now  = sum(p.ships for p in planets if p.owner==pid)
        prod_rate  = sum(p.production for p in planets if p.owner==pid)
        fleets_now = sum(f.ships for f in fleets if f.owner==pid)
        # Simple linear: ships + production*turns + fleets in transit
        projections[pid] = ships_now + prod_rate*turns_left + fleets_now

    return projections


def endgame_steal_target(planets, fleets, player, mine, enemy,
                          current_step, done, enroute):
    """
    If we project to lose at turn 500:
    Find the one enemy planet whose capture closes the deficit.
    Returns (planet, priority_score) or None.
    """
    if current_step < 200: return None  # too early for endgame thinking
    if not mine or not enemy: return None

    proj = project_turn500(planets, fleets, player, current_step)
    my_proj = proj.get(player, 0)

    best_enemy_proj = max(
        (proj.get(p.owner, 0) for p in enemy if p.owner >= 0),
        default=0
    )

    if my_proj >= best_enemy_proj * 0.92: return None  # we're fine

    deficit = best_enemy_proj - my_proj
    turns_left = MS - current_step

    # Find planet whose production × turns_left ≈ closes the deficit
    # Best steal = highest production that is capturable
    candidates = [p for p in enemy
                  if p.id not in done and p.id not in enroute]
    if not candidates: return None

    # Sort by how much they contribute to closing deficit
    def steal_value(p):
        prod_gain = p.production * turns_left  # we gain this
        prod_deny = p.production * turns_left  # enemy loses this
        total_swing = prod_gain + prod_deny
        return total_swing

    best = max(candidates, key=steal_value)
    score = steal_value(best)
    if score > deficit * 0.3:  # worth at least 30% of deficit
        return best, score
    return None


# ═════════════════════════════════════════════════════════════════════════════
# WEAPON 17: GRAVITY SLINGSHOT PATH
# Instead of shortest path, route fleet on tangent arc around the sun
# Arrives from enemy's BLIND SIDE (opposite direction they expect)
# Also useful when direct path is blocked by sun
# ═════════════════════════════════════════════════════════════════════════════
def slingshot_angle(ox, oy, tx, ty, clockwise=True):
    """
    Compute launch angle for a path that skirts the sun's edge.
    The fleet goes 'around' the sun to hit from the other side.
    Returns (angle, is_valid)
    """
    # Target angle from origin
    direct_angle = math.atan2(ty-oy, tx-ox)

    # Slingshot: offset by 60-90 degrees to go around
    offset = math.radians(75) * (1 if clockwise else -1)
    sling_angle = direct_angle + offset

    # Check it doesn't hit the sun itself
    d_total = d2(ox,oy,tx,ty)
    if sun_ok(ox,oy,sling_angle,d_total*1.4):
        return sling_angle, True
    return direct_angle, False


def slingshot_move(src, tgt, av, n_ships):
    """
    Returns (angle, is_slingshot) for a slingshot or direct path.
    Tries clockwise and counter-clockwise slingshot.
    """
    # Try direct first
    a_direct,dd,eta = icp(src.x,src.y,tgt,av,n_ships)
    sa_direct,ok_direct = safe(src.x,src.y,a_direct,dd)

    # Try slingshot CW
    a_cw,ok_cw = slingshot_angle(src.x,src.y,tgt.x,tgt.y,clockwise=True)
    a_ccw,ok_ccw = slingshot_angle(src.x,src.y,tgt.x,tgt.y,clockwise=False)

    # Prefer slingshot if direct path hits sun
    if not ok_direct and ok_cw:
        return a_cw, True
    if not ok_direct and ok_ccw:
        return a_ccw, True

    # Use slingshot occasionally even when direct is available
    # (surprise attack from blind side — 25% of high-value strikes)
    import random
    if ok_cw and n_ships > 20 and random.random() < 0.25:
        return a_cw, True

    return sa_direct, False


# ═════════════════════════════════════════════════════════════════════════════
# v3 weapons (all preserved)
# ═════════════════════════════════════════════════════════════════════════════
def exact_capture_n(target, av, sx, sy, buf=1.07):
    lo,hi=1,max(target.ships*2+20,30)
    for _ in range(16):
        mid=(lo+hi)//2
        _,_,eta=icp(sx,sy,target,av,mid)
        garrison=target.ships+target.production*eta
        if mid>garrison*buf: hi=mid
        else: lo=mid+1
    return hi

def roi_ok(n,prod,eta,rem,status):
    if prod==0: return True
    pb=n/max(1,prod)
    thr=120 if status=='losing' else 80
    return pb<thr and (rem-eta)>pb

def phase_mult(planet,mine,av):
    if not inn(planet) or not mine: return 1.0
    r=d2(planet.x,planet.y,SX,SY); a0=math.atan2(planet.y-SY,planet.x-SX)
    fx=SX+r*math.cos(a0+av*15); fy=SY+r*math.sin(a0+av*15)
    cx=sum(p.x for p in mine)/len(mine); cy=sum(p.y for p in mine)/len(mine)
    if d2(fx,fy,cx,cy)<d2(planet.x,planet.y,cx,cy)*0.85: return 1.65
    if d2(fx,fy,cx,cy)>d2(planet.x,planet.y,cx,cy)*1.15: return 0.72
    return 1.0

def compute_status(planets,fleets,mine,player):
    if not mine: return 'losing',1.0
    cx=sum(p.x for p in mine)/len(mine); cy=sum(p.y for p in mine)/len(mine)
    mp=(sum(p.ships+p.production*25 for p in mine)
        +sum(f.ships for f in fleets if f.owner==player))
    ep=0
    for p in planets:
        if p.owner<0 or p.owner==player: continue
        prx=max(0,1-d2(p.x,p.y,cx,cy)/60)
        ep+=(p.ships+p.production*25)*(0.5+prx)
    for f in fleets:
        if f.owner==player or f.owner<0: continue
        dx,dy=math.cos(f.angle),math.sin(f.angle)
        if (cx-f.x)*dx+(cy-f.y)*dy>0: ep+=f.ships*0.8
    r=mp/max(1,ep)
    if r>=1.15: return 'winning',0.5
    if r>=0.85: return 'even',0.75
    return 'losing',1.0

def garrison(planet,status,aggr,incoming=0):
    if incoming>0: return int(incoming*1.12)+5
    base=max(3,planet.production*2)
    if status=='winning': return base
    if aggr>0.85: return max(3,base//2)
    return base

def mirror_moves(neutral,efleets,mine,av,used,done,enroute):
    out=[]
    for tgt in neutral:
        if tgt.id in done or tgt.id in enroute: continue
        inc=[]
        for f in efleets:
            _,dd,eta=icp(f.x,f.y,tgt,av,f.ships)
            if dd<tgt.radius+4 and eta<60: inc.append((eta,f.ships))
        if not inc: continue
        e_eta,e_ships=sorted(inc)[0]
        after=tgt.ships+tgt.production*e_eta
        if e_ships<=after: continue
        left=max(1,e_ships-after)
        n=int(left*1.1)+tgt.production*2+2
        for src in sorted(mine,key=lambda p:d2(p.x,p.y,tgt.x,tgt.y)):
            sp=src.ships-used.get(src.id,0)-4
            if sp<n: continue
            _,_,our=icp(src.x,src.y,tgt,av,n)
            if our<e_eta+0.5 or our>e_eta+8: continue
            a,dd3,_=icp(src.x,src.y,tgt,av,n); sa,ok=safe(src.x,src.y,a,dd3)
            if ok: out.append((tgt.production*5+e_ships*0.3,src.id,sa,n,tgt.id)); break
    out.sort(key=lambda x:-x[0]); return out

def desync_attack(sources,target,av,used):
    if len(sources)<2: return []
    arrivals=[]
    for src in sources[:3]:
        sp=src.ships-used.get(src.id,0)-3
        if sp<4: continue
        _,dd,eta=icp(src.x,src.y,target,av,sp)
        arrivals.append((eta,src,sp,dd))
    if len(arrivals)<2: return []
    arrivals.sort(key=lambda x:x[0])
    t_eta=arrivals[len(arrivals)//2][0]
    garrison_arr=target.ships+target.production*t_eta
    mvs=[]; tot=0
    for eta,src,sp,dd in arrivals:
        if eta>t_eta+3: continue
        n=max(4,sp//len(arrivals))
        if src.ships-used.get(src.id,0)-n<3: continue
        a2,dd2,_=icp(src.x,src.y,target,av,n)
        sa,ok=safe(src.x,src.y,a2,dd2)
        if ok: mvs.append((src.id,sa,n)); tot+=n
    if tot>garrison_arr*1.05 and len(mvs)>=2: return mvs
    return []

def find_massing(enemy):
    out=[]
    for p in enemy:
        h=_hist.get(p.id,[])
        if len(h)<4: continue
        if p.ships-h[-4]>p.production*4*0.7 and p.ships>30:
            out.append((p,p.ships*p.production))
    out.sort(key=lambda x:-x[1]); return out

def retro_targets(planets,fleets,player):
    out=[]
    for p in planets:
        if p.owner<0 or p.owner==player: continue
        dep=sum(f.ships for f in fleets
                if f.owner==p.owner and d2(f.x,f.y,p.x,p.y)<12)
        if dep<8: continue
        r=dep/max(1,p.ships+dep)
        if r>=0.28: out.append((p,r*p.production*4+dep*0.3))
    out.sort(key=lambda x:-x[1]); return out

def suffocation_moves(enemy,mine,av,used,step,status):
    if step<50: return []
    out=[]
    for tgt in sorted([p for p in enemy if p.production>=3],
                       key=lambda p:-p.production)[:2]:
        h=_hist.get(tgt.id,[])
        interval=15; n=3
        if len(h)>=4:
            gr=(tgt.ships-h[-4])/4
            if gr>tgt.production*1.3: interval=8;  n=5
            elif gr<tgt.production*0.5: interval=25
        if step%interval not in (0,1): continue
        src=min([p for p in mine if p.ships-used.get(p.id,0)-5>=n],
                key=lambda p:d2(p.x,p.y,tgt.x,tgt.y),default=None)
        if src is None: continue
        a,dd,_=icp(src.x,src.y,tgt,av,n); sa,ok=safe(src.x,src.y,a,dd)
        if ok: out.append((src.id,sa,n))
    return out

def feint_moves(enemy,mine,av,used,step,done):
    if step<40 or step%20!=0 or not enemy or not mine: return []
    cx=sum(p.x for p in mine)/len(mine); cy=sum(p.y for p in mine)/len(mine)
    far=[p for p in enemy if p.id not in done and p.id not in _feints]
    if not far: return []
    tgt=max(far,key=lambda p:d2(p.x,p.y,cx,cy))
    src=max([p for p in mine if p.ships-used.get(p.id,0)-8>=2],
            key=lambda p:p.ships-used.get(p.id,0),default=None)
    if src is None: return []
    a,dd,_=icp(src.x,src.y,tgt,av,2); sa,ok=safe(src.x,src.y,a,dd)
    if ok: _feints[tgt.id]=step; return [(src.id,sa,2,tgt.id)]
    return []

def find_chokepoints(planets,mine,enemy,neutral,player):
    if not enemy: return []
    ecx=sum(p.x for p in enemy)/len(enemy); ecy=sum(p.y for p in enemy)/len(enemy)
    mcx=sum(p.x for p in mine)/len(mine);   mcy=sum(p.y for p in mine)/len(mine)
    cands=neutral+[p for p in enemy if p.ships<15]
    out=[]
    for p in cands:
        if d2(p.x,p.y,mcx,mcy)>70: continue
        sc=(50/(d2(p.x,p.y,ecx,ecy)+1))*(p.production+1)*(1/(d2(p.x,p.y,mcx,mcy)+5))
        out.append((p,sc))
    out.sort(key=lambda x:-x[1]); return out[:3]

# ═════════════════════════════════════════════════════════════════════════════
# MAIN AGENT
# ═════════════════════════════════════════════════════════════════════════════
def orbital_strategist(obs):
    global _hist,_feints,_bait_id,_coalition,av

    if isinstance(obs,dict):
        pl=obs.get('player',0); rp=obs.get('planets',[])
        rf=obs.get('fleets',[]); av=obs.get('angular_velocity',0.0366)
        stp=obs.get('step',0)
    else:
        pl=obs.player; rp=obs.planets; rf=obs.fleets
        av=obs.angular_velocity; stp=getattr(obs,'step',0)

    try:
        from kaggle_environments.envs.orbit_wars.orbit_wars import Planet as NP,Fleet as NF
        planets=[NP(*p) for p in rp]; fleets=[NF(*f) for f in rf]
    except Exception:
        planets=[_P(*p) for p in rp]; fleets=[_F(*f) for f in rf]

    mine    =[p for p in planets if p.owner==pl]
    neutral =[p for p in planets if p.owner<0]
    enemy   =[p for p in planets if p.owner>=0 and p.owner!=pl]
    others  =enemy+neutral
    if not mine or not others: return []

    rem=MS-stp; moves=[]; used={}; done=set()
    def avail(p): return p.ships-used.get(p.id,0)
    def rsv(pid,n): used[pid]=used.get(pid,0)+n

    # Update history
    for p in planets:
        if p.owner>=0:
            _hist[p.id].append(p.ships)
            if len(_hist[p.id])>8: _hist[p.id].pop(0)

    status,aggr = compute_status(planets,fleets,mine,pl)
    e_fleets    = [f for f in fleets if f.owner!=pl and f.owner>=0]
    inner_enemy = [p for p in enemy if inn(p)]

    # Incoming threats
    incoming={}
    for f in fleets:
        if f.owner==pl: continue
        for p in mine:
            _,dd,_=icp(f.x,f.y,p,av,f.ships)
            if dd<p.radius+spd(f.ships)*1.5+1:
                incoming[p.id]=incoming.get(p.id,0)+f.ships

    # ── DEFENSE ───────────────────────────────────────────────────────────
    for p in mine:
        thr=incoming.get(p.id,0)
        if thr==0: continue
        need=garrison(p,status,aggr,thr); deficit=need-avail(p)
        if deficit<=0: continue
        for src in sorted([s for s in mine if s.id!=p.id
                           and avail(s)-garrison(s,status,aggr)>4],
                          key=lambda s:d2(s.x,s.y,p.x,p.y))[:3]:
            snd=min(avail(src)-garrison(src,status,aggr),deficit)
            if snd<=0: continue
            a,dd,_=icp(src.x,src.y,p,av,snd); sa,ok=safe(src.x,src.y,a,dd)
            if ok: moves.append([src.id,sa,snd]); rsv(src.id,snd); deficit-=snd
            if deficit<=0: break

    # En-route
    enroute=set()
    for f in fleets:
        if f.owner!=pl: continue
        for t in others:
            _,dd,eta=icp(f.x,f.y,t,av,f.ships)
            if dd<t.radius+4 and eta<70: enroute.add(t.id)

    # ── W15: COALITION EXPLOIT ────────────────────────────────────────────
    at_war = detect_coalition_war(fleets,planets,pl)

    # ── W13: BAIT PLANET TRAP ─────────────────────────────────────────────
    if stp > 80 and stp % 30 == 0 and len(mine) >= 4:
        bp = select_bait_planet(mine,enemy,av)
        if bp: _bait_id = bp.id

    bait_planet = next((p for p in mine if p.id==_bait_id),None)
    if bait_planet and bait_planet.ships > 3:
        # Drain it to 3 — looks tempting
        drain=bait_planet.ships-3
        if drain>0:
            hub=max([p for p in mine if p.id!=bait_planet.id
                     and avail(p)>5],
                    key=lambda p:p.production, default=None)
            if hub:
                a,dd,_=icp(bait_planet.x,bait_planet.y,hub,av,drain)
                sa,ok=safe(bait_planet.x,bait_planet.y,a,dd)
                if ok:
                    moves.append([bait_planet.id,sa,drain])
                    rsv(bait_planet.id,drain)

    # Bait counter-strike
    if bait_planet:
        biggest_bait_fleet=None; best_dd=1e9
        for f in e_fleets:
            _,dd,eta=icp(f.x,f.y,bait_planet,av,f.ships)
            if dd<bait_planet.radius+4 and eta<50:
                if dd<best_dd: best_dd=dd; biggest_bait_fleet=f
        if biggest_bait_fleet:
            # Find enemy source (closest enemy planet to fleet origin)
            src_planet=min(
                [p for p in enemy if p.id not in done],
                key=lambda p:d2(p.x,p.y,biggest_bait_fleet.x,biggest_bait_fleet.y),
                default=None)
            if src_planet and src_planet.id not in enroute:
                attacker=max([p for p in mine if avail(p)-garrison(p,status,aggr)>5],
                             key=lambda p:avail(p)-garrison(p,status,aggr),default=None)
                if attacker:
                    n=exact_capture_n(src_planet,av,attacker.x,attacker.y)
                    sp=avail(attacker)-garrison(attacker,status,aggr)
                    if sp>=n:
                        # W17: use slingshot for surprise
                        sa,is_sling=slingshot_move(attacker,src_planet,av,n)
                        moves.append([attacker.id,sa,n])
                        rsv(attacker.id,n); done.add(src_planet.id)

    # ── W12: PHASE SPACE SYNC ─────────────────────────────────────────────
    if inner_enemy and stp < 450:
        for sid,sa,n,tid in phase_sync_moves(inner_enemy,mine,av,used,done,enroute,stp,status):
            src=next((p for p in mine if p.id==sid),None)
            if src and avail(src)-garrison(src,status,aggr)>=n:
                moves.append([sid,sa,n]); rsv(sid,n); done.add(tid)

    # ── W1: MIRROR COUNTER ────────────────────────────────────────────────
    for sc,sid,sa,n,tid in mirror_moves(neutral,e_fleets,mine,av,used,done,enroute)[:2]:
        src=next((p for p in mine if p.id==sid),None)
        if src and avail(src)>=n+garrison(src,status,aggr):
            moves.append([sid,sa,n]); rsv(sid,n); done.add(tid)

    # ── W2: FEINT ─────────────────────────────────────────────────────────
    for fm in feint_moves(enemy,mine,av,used,stp,done):
        src=next((p for p in mine if p.id==fm[0]),None)
        if src and avail(src)>=fm[2]+garrison(src,status,aggr):
            moves.append([fm[0],fm[1],fm[2]]); rsv(fm[0],fm[2])

    # ── W5: CHOKEPOINTS ───────────────────────────────────────────────────
    for choke,_ in find_chokepoints(planets,mine,enemy,neutral,pl)[:1]:
        if choke.id in done or choke.id in enroute: continue
        bsrc=min([p for p in mine if avail(p)-garrison(p,status,aggr)>5],
                 key=lambda p:d2(p.x,p.y,choke.x,choke.y),default=None)
        if bsrc is None: continue
        n=exact_capture_n(choke,av,bsrc.x,bsrc.y)
        if avail(bsrc)-garrison(bsrc,status,aggr)<n: continue
        _,_,eta=icp(bsrc.x,bsrc.y,choke,av,n)
        if not roi_ok(n,choke.production,eta,rem,status): continue
        a,dd,_=icp(bsrc.x,bsrc.y,choke,av,n); sa,ok=safe(bsrc.x,bsrc.y,a,dd)
        if ok: moves.append([bsrc.id,sa,n]); rsv(bsrc.id,n); done.add(choke.id)

    # ── W6: SUFFOCATION ───────────────────────────────────────────────────
    for sid,sa,n in suffocation_moves(enemy,mine,av,used,stp,status):
        src=next((p for p in mine if p.id==sid),None)
        if src and avail(src)>=n+garrison(src,status,aggr):
            moves.append([sid,sa,n]); rsv(sid,n)

    # ── W9: PRE-EMPTIVE ───────────────────────────────────────────────────
    for tgt,_ in find_massing(enemy)[:1]:
        if tgt.id in done or tgt.id in enroute: continue
        bsrc=max([p for p in mine if avail(p)-garrison(p,status,aggr)>8],
                 key=lambda p:avail(p)-garrison(p,status,aggr),default=None)
        if bsrc is None: continue
        n=exact_capture_n(tgt,av,bsrc.x,bsrc.y)
        if avail(bsrc)-garrison(bsrc,status,aggr)<n: continue
        _,_,eta=icp(bsrc.x,bsrc.y,tgt,av,n)
        if not roi_ok(n,tgt.production,eta,rem,status): continue
        sa,is_sling=slingshot_move(bsrc,tgt,av,n)   # W17
        moves.append([bsrc.id,sa,n]); rsv(bsrc.id,n); done.add(tgt.id)

    # ── W10: RETROGRADE ───────────────────────────────────────────────────
    for rp_t,_ in retro_targets(planets,fleets,pl)[:2]:
        if rp_t.id in done or rp_t.id in enroute: continue
        bsrc=None; bn=0; bsa=0.0; bdd=1e9
        for src in mine:
            sp=avail(src)-garrison(src,status,aggr)
            if sp<4: continue
            n=exact_capture_n(rp_t,av,src.x,src.y)
            if sp<n: continue
            _,_,eta=icp(src.x,src.y,rp_t,av,n)
            if not roi_ok(n,rp_t.production,eta,rem,status): continue
            a2,dd2,_=icp(src.x,src.y,rp_t,av,n); sa,ok=safe(src.x,src.y,a2,dd2)
            if not ok: continue
            if bsrc is None or dd2<bdd: bsrc,bn,bsa,bdd=src,n,sa,dd2
        if bsrc: moves.append([bsrc.id,bsa,bn]); rsv(bsrc.id,bn); done.add(rp_t.id)

    # ── W7: DESYNC on fortified ───────────────────────────────────────────
    for tgt in sorted([t for t in enemy if t.id not in done
                       and t.id not in enroute
                       and t.production>=3 and t.ships>25],
                      key=lambda t:-t.production*t.ships)[:1]:
        srcs=[p for p in mine if avail(p)-garrison(p,status,aggr)>6]
        dm=desync_attack(srcs,tgt,av,used)
        if dm:
            for sid2,sa2,n2 in dm:
                moves.append([sid2,sa2,n2]); rsv(sid2,n2)
            done.add(tgt.id)

    # ── W16: ENDGAME PROJECTION ───────────────────────────────────────────
    steal_result = endgame_steal_target(planets,fleets,pl,mine,
                                        enemy,stp,done,enroute)
    if steal_result:
        steal_tgt, _ = steal_result
        bsrc=max([p for p in mine if avail(p)-garrison(p,status,aggr)>5],
                 key=lambda p:avail(p)-garrison(p,status,aggr),default=None)
        if bsrc:
            n=exact_capture_n(steal_tgt,av,bsrc.x,bsrc.y)
            if avail(bsrc)-garrison(bsrc,status,aggr)>=n:
                sa,_=slingshot_move(bsrc,steal_tgt,av,n)   # W17
                moves.append([bsrc.id,sa,n]); rsv(bsrc.id,n); done.add(steal_tgt.id)

    # ── MAIN ROI SCORING (W3 + W11 + W15) ────────────────────────────────
    cands=[]
    for src in mine:
        spare=avail(src)-garrison(src,status,aggr)
        if spare<4: continue
        for tgt in others:
            if tgt.id in done or tgt.id in enroute: continue
            n=exact_capture_n(tgt,av,src.x,src.y)
            if n>spare: continue
            a2,dd2,eta2=icp(src.x,src.y,tgt,av,n)
            sa,ok=safe(src.x,src.y,a2,dd2)
            if not ok: continue
            if not roi_ok(n,tgt.production,eta2,rem,status): continue
            tw=max(0,rem-eta2); prod=tgt.production
            score=(prod**2)*10*tw+prod*tw
            score*=phase_mult(tgt,mine,av)           # W11
            score*=coalition_expansion_bonus(tgt,at_war,mine)  # W15
            if tgt.owner>=0:
                score*=1.5
                ep2=sum(p.production for p in planets if p.owner==tgt.owner)
                if ep2>sum(p.production for p in mine)*1.1: score*=1.3
            if tgt.ships<=tgt.production*2+3: score*=1.9
            score-=dd2*0.4+n*0.25
            if status=='losing': score=score*1.4 if prod>=3 else score*0.6
            cands.append((score,src,tgt,n,sa,dd2))

    cands.sort(key=lambda x:-x[0])
    max_atk=5 if(stp<90 or status=='losing') else 4
    atks=0
    for score,src,tgt,n,sa,dd in cands:
        if atks>=max_atk: break
        if tgt.id in done or tgt.id in enroute: continue
        if avail(src)-garrison(src,status,aggr)<n: continue
        # W17: slingshot 20% of main attacks
        sa_final,_=slingshot_move(src,tgt,av,n)
        moves.append([src.id,sa_final,n]); rsv(src.id,n); done.add(tgt.id); atks+=1

    # ── SWEEP: zero idle ships ─────────────────────────────────────────────
    for src in sorted(mine,key=lambda p:-avail(p)):
        spare=avail(src)-garrison(src,status,aggr)
        if spare<5: continue
        best=None; bsc=-1e9
        for tgt in others:
            if tgt.id in done: continue
            n=exact_capture_n(tgt,av,src.x,src.y)
            if n>spare: continue
            a2,dd2,_=icp(src.x,src.y,tgt,av,n)
            sa,ok=safe(src.x,src.y,a2,dd2)
            if not ok: continue
            sc=(tgt.production**2)/(dd2+1)*phase_mult(tgt,mine,av)
            if sc>bsc: bsc=sc; best=(src.id,sa,n,tgt.id)
        if best:
            moves.append([best[0],best[1],best[2]])
            rsv(best[0],best[2]); done.add(best[3])

    return moves

agent = orbital_strategist


## ✅ Cell 9 — Verify


In [ ]:
import importlib.util, collections as _c
_hist=_c.defaultdict(list); _feints={}; _bait_id=None; _coalition=_c.defaultdict(int)
spec=importlib.util.spec_from_file_location('main','main.py')
mod=importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
sub=mod.agent
print(f'✅ main.py OK — {sub.__name__}')
ev=make('orbit_wars',debug=False)
ev.run([sub,v1_agent,'random',v1_agent])
fr=[s.reward for s in ev.steps[-1]]
print(f'Rewards: {fr}')
print('🏆 WINS!' if fr[0]==1 else '✅ Runs correctly')
